In [ ]:
import json
import os
import pickle
from pathlib import Path

import torch
from dotenv import dotenv_values

In [ ]:
from config import (
    CHUNK_OVERLAP,
    CHUNK_SIZE,
    COLLECTION_NAME,
    EMBEDDING_MODEL,
    LANGFUSE_HOST,
    LANGFUSE_PUBLIC_KEY,
    LANGFUSE_SECRET_KEY,
    SEMANTIC_THRESHOLD,
    SPLITTER_TYPE,
    SPLITTING_MODEL,
)

In [ ]:
config = dotenv_values(".env")
OPENAI_API_KEY = config.get("OPENAI_API_KEY")

os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY
os.environ["LANGFUSE_HOST"] = LANGFUSE_HOST

In [ ]:
from langfuse import get_client
langfuse = get_client()

In [ ]:
print(f"CUDA available: {torch.cuda.is_available()}")
torch.set_float32_matmul_precision("high")

## Configuration

In [ ]:
print(f"Splitter type: {SPLITTER_TYPE}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Collection name: {COLLECTION_NAME}")

if SPLITTER_TYPE == "semantic":
    print(f"Splitting model: {SPLITTING_MODEL}")
    print(f"Threshold: {SEMANTIC_THRESHOLD}")
else:
    print(f"Chunk size: {CHUNK_SIZE}")
    print(f"Chunk overlap: {CHUNK_OVERLAP}")

## Initialize Text Splitter

In [ ]:
if SPLITTER_TYPE == "semantic":
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_experimental.text_splitter import SemanticChunker
    
    splitting_embeddings = HuggingFaceEmbeddings(
        model_name=SPLITTING_MODEL,
        model_kwargs={"device": "cuda"},
    )
    
    text_splitter = SemanticChunker(
        embeddings=splitting_embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=SEMANTIC_THRESHOLD,
    )
    print(f"Initialized SemanticChunker")

elif SPLITTER_TYPE == "recursive":
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )
    print(f"Initialized RecursiveCharacterTextSplitter")

else:
    raise ValueError(f"Unknown splitter type: {SPLITTER_TYPE}")

## Load Documents

In [ ]:
from langchain_community.document_loaders import TextLoader

docs_dir = Path("/home/bartek/Kod/PD/praca_dyplomowa/dane/texts/cleaned_texts")
documents = []

for doc_dir in docs_dir.glob("*"):
    txt_files = list(doc_dir.glob("LLM_clean_text*.txt"))
    if not txt_files:
        continue

    loader = TextLoader(txt_files[0], encoding="utf-8")
    loaded_docs = loader.load()

    json_file = doc_dir / "meta_data.json"
    if not json_file.exists():
        continue

    with json_file.open("r", encoding="utf-8") as f:
        meta_from_json = json.load(f)

    for doc in loaded_docs:
        doc.metadata["title"] = meta_from_json["title"]
        doc.metadata["language"] = meta_from_json["language"]
        doc.metadata["url"] = meta_from_json["url"]
        doc.metadata["urldate"] = meta_from_json["urldate"]

    documents.extend(loaded_docs)

print(f"Loaded {len(documents)} documents")

## Split Documents

In [ ]:
split_docs = text_splitter.split_documents(documents)
print(f"Created {len(split_docs)} chunks")

In [ ]:
chunk_lengths = [len(doc.page_content) for doc in split_docs]
sorted_lengths = sorted(chunk_lengths)

chunk_stats = {
    "total_chunks": len(split_docs),
    "total_documents": len(documents),
    "avg_chunk_size": int(sum(chunk_lengths) / len(chunk_lengths)),
    "max_chunk_size": max(chunk_lengths),
    "min_chunk_size": min(chunk_lengths),
    "median_chunk_size": sorted_lengths[len(sorted_lengths) // 2],
    "p25_chunk_size": sorted_lengths[len(sorted_lengths) // 4],
    "p75_chunk_size": sorted_lengths[3 * len(sorted_lengths) // 4],
    "p95_chunk_size": sorted_lengths[int(0.95 * len(sorted_lengths))],
}

print("=" * 60)
print("CHUNK STATISTICS")
print(f"Total documents: {chunk_stats['total_documents']}")
print(f"Total chunks: {chunk_stats['total_chunks']}")
print(f"Chunks per document: {chunk_stats['total_chunks'] / chunk_stats['total_documents']:.1f}")
print(f"\nChunk sizes (characters):")
print(f"  Min:    {chunk_stats['min_chunk_size']:>6}")
print(f"  25th %: {chunk_stats['p25_chunk_size']:>6}")
print(f"  Median: {chunk_stats['median_chunk_size']:>6}")
print(f"  Average:{chunk_stats['avg_chunk_size']:>6}")
print(f"  75th %: {chunk_stats['p75_chunk_size']:>6}")
print(f"  95th %: {chunk_stats['p95_chunk_size']:>6}")
print(f"  Max:    {chunk_stats['max_chunk_size']:>6}")
print("=" * 60)

## Save Splits

In [ ]:
splitter_suffix = f"{SPLITTER_TYPE}"
if SPLITTER_TYPE == "semantic":
    splitter_suffix += f"_{SPLITTING_MODEL.split('/')[-1]}_t{SEMANTIC_THRESHOLD}"
else:
    splitter_suffix += f"_s{CHUNK_SIZE}_o{CHUNK_OVERLAP}"

splits_file = Path(f"/home/bartek/Kod/PD/praca_dyplomowa/dane/vectordb/{splitter_suffix}_split_documents.pkl")

with splits_file.open("wb") as f:
    pickle.dump(split_docs, f)

print(f"Saved splits to: {splits_file}")

## Clear GPU Memory (if semantic splitter)

In [ ]:
if SPLITTER_TYPE == "semantic":
    del splitting_embeddings
    del text_splitter
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print("Cleared CUDA cache")

## Create Embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cuda", "trust_remote_code": True},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 8},
)

In [ ]:
from langchain_chroma import Chroma

chroma_db_path = Path(f"/home/bartek/Kod/PD/praca_dyplomowa/dane/vectordb/{splitter_suffix}_{EMBEDDING_MODEL.split('/')[-1]}_chroma_db")

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=str(chroma_db_path),
)

print(f"Created vectorstore at: {chroma_db_path}")

## Log to Langfuse

In [ ]:
trace = langfuse.trace(
    name="embedding_pipeline",
    metadata={
        "splitter_type": SPLITTER_TYPE,
        "embedding_model": EMBEDDING_MODEL,
        "collection": COLLECTION_NAME,
        "splitting_model": SPLITTING_MODEL if SPLITTER_TYPE == "semantic" else None,
        "threshold": SEMANTIC_THRESHOLD if SPLITTER_TYPE == "semantic" else None,
        "chunk_size": CHUNK_SIZE if SPLITTER_TYPE == "recursive" else None,
        "chunk_overlap": CHUNK_OVERLAP if SPLITTER_TYPE == "recursive" else None,
    },
    output={
        "status": "complete",
        "splits_file": str(splits_file),
        "vectorstore_path": str(chroma_db_path),
        **chunk_stats,
    },
)

langfuse.flush()
print("Logged to Langfuse")

In [ ]:
print("Done!")